# Modeling the Control Policy (When to Intervene)

**MODELING notebook — pre-generation gating, the dev-to-eval methodology, policy value, and ceiling recovery**

This is a **modeling** notebook: every section introduces one modeling choice and evaluates it.

Every section follows *Question → What we do → Figure/Table → Reading → Artifact → Caveat*.
Each code cell states what it does and each output is interpreted in the following
cell, so a reader with no access to the code can follow the reasoning. All numbers are
read from immutable artifacts in `results/` through `paper_lib`; missing optional
experiments print `PENDING` with their producer command instead of failing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import paper_lib as L

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 200)
np.random.seed(42)

PRIMARY_RUN = "M4_intersection_dev_conditioned_continuous"
BLIND_RUN = "M4_intersection_dev_blind_continuous"
GATE_FEATURE_COLS = ["top1_faiss", "top5_mean_faiss", "top5_min_faiss", "top5_std_faiss",
                     "retrieval_margin", "top5_spread", "retrieval_overlap", "max_lift",
                     "mean_lift", "n_positive_lifts", "n_negative_lifts", "lift_conflict",
                     "evidence_density", "query_desc_len", "query_title_len"]
print("Project root:", L.ROOT)

## The control problem

Even a calibrated prior has per-ticket variance: some tickets are helped, some harmed. A **control
policy** decides, before generating, whether to apply feedback to a given ticket. The train/dev/eval
split is designed for exactly this: feedback is *built* from train, every configuration is
*developed* on dev, patterns that hold on dev are *modeled*, and the resulting policy is *tested once*
on eval. This notebook builds and evaluates that policy on dev; notebook 06 confirms on eval.

## Why a control policy can only help under some conditions

A policy replaces feedback with the baseline on tickets it closes. If always-on has a positive mean,
closing tickets removes benefit on average, so a policy helps only if it can rank the **sign** of the
per-ticket effect better than chance. Formally, a policy beats always-on when the area under its
prediction curve exceeds a crossover near 0.5; at exactly chance it is worse, because the average
effect is positive. Under ticket-only feedback, always-on is *negative*, so closing harmful tickets
helps and the policy has much more room. This asymmetry is the key to everything below.

### What can the policy observe before generation?

**What we do.** Only pre-generation features: retrieval confidence, margin, spread, overlap, lift statistics, evidence density, and query lengths. No reference reply, no generated answer.

**Artifact.** `src/gate/features.py; results/blend_eb/gate_features_train.parquet`

**Caveat.** Team/class identity is excluded so the policy is not dataset-specific.

*What this cell does.* Load the gate feature table and show summary statistics for the features available at decision time.

In [ ]:
feat = pd.read_parquet(L.RESULTS / "blend_eb" / "gate_features_train_M4_intersection.parquet")
cols = [c for c in GATE_FEATURE_COLS if c in feat.columns]
display(feat[cols].describe().T.round(3))

**Reading.** The features are all quantities a retrieval system knows before calling the generator:
how confident the baseline is, how separated the top candidates are, how much feedback evidence
exists, and how large the potential bonus is. Because none of them is team- or class-specific, a
policy learned on them can transfer across organizations.

### Does a policy help under ticket-only feedback?

**What we do.** Use the fixed general gate study (grouped CV by ticket, thresholds frozen on dev) and its counterfactual decomposition.

**Artifact.** `results/gate_study_general/{learned_gate.csv,learned_gate_decomposition.csv}`

**Caveat.** The definitive study; the earlier per-run pilot numbers were superseded by the methodology fix.

*What this cell does.* Show the eval AUC, the dev-frozen policy value, and the counterfactual decomposition for the two blind Laplace configurations.

In [ ]:
learned = L.load_gate_study_learned()
decomp = L.load_gate_study_decomposition()
blind = learned[(learned["protocol"] == "blind") & (learned["config_key"].str.contains("laplace"))]
display(blind[["config_key", "dev_threshold", "eval_auc", "eval_auc_ci_lower", "eval_auc_ci_upper",
               "always_on", "eval_policy", "eval_policy_ci_lower", "eval_policy_ci_upper",
               "pct_open", "gain_vs_always_on", "ceiling_recovery"]].round(4))
display(decomp[decomp["config_key"].str.contains("laplace")].round(4))
fig, ax = plt.subplots(figsize=(9, 4.6))
sub = learned[learned["config_key"].str.contains("laplace")].copy()
sub["label"] = sub["config_key"] + " / " + sub["protocol"]
sns.barplot(data=sub.melt(id_vars=["label"], value_vars=["always_on", "eval_policy", "oracle"]),
            x="label", y="value", hue="variable", ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Mean generated delta", title="Gate policy value (eval, dev-frozen threshold)")
plt.tight_layout(); L.savefig("05_policy_blind", run_ids=[]); plt.show()

**Reading.** Under uncalibrated ticket-only feedback the gate clearly helps: always-on is −0.038
and the dev-frozen policy is +0.001 to +0.002, recovering 0.54–0.56 of the oracle ceiling. The
decomposition shows why: the gate closes 152–235 tickets whose counterfactual mean delta is
−0.07 to −0.10, i.e. a genuinely harmful subset. The AUC is modest (0.59–0.61), but because the
alternative is strongly negative even a modest ranking is valuable. This is the deployment setting
where the control policy earns its place.

### Does a policy help under resolution-informed feedback?

**What we do.** Repeat on the conditioned configurations with the same fixed methodology.

**Artifact.** `results/gate_study_general/learned_gate.csv`

**Caveat.** Same features and labels; only the feedback protocol changes.

*What this cell does.* Compare AUC, dev-frozen policy value and recovery across protocols.

In [ ]:
sub = learned[learned["config_key"].str.contains("laplace")].copy()
sub["label"] = sub["config_key"] + " / " + sub["protocol"]
fig, ax = plt.subplots(figsize=(9, 4.6))
sns.barplot(data=sub, x="label", y="eval_auc", hue="protocol", ax=ax)
ax.axhline(0.5, color="black", ls="--", lw=1); ax.set(ylabel="Eval AUC", title="Gate predictability by protocol")
plt.tight_layout(); L.savefig("05_auc_by_protocol", run_ids=[]); plt.show()
display(sub[["label", "eval_auc", "always_on", "eval_policy", "gain_vs_always_on", "ceiling_recovery"]].round(4))

**Reading.** Under resolution-informed feedback the gate is essentially neutral: the frozen
policy matches always-on within ±0.0005 (team AUC 0.56; intersection AUC 0.66 but no recoverable
headroom because always-on is already positive). The decomposition shows the closed set is mostly
no-op tickets with a slightly positive mean, so closing them cannot help. The honest conclusion is
that a control policy is justified only when feedback reliability is low and harm is concentrated.

### Is a learned policy worth it, or does a simple rule suffice?

**What we do.** Compare the learned gate against the best single-feature static rule derived on dev.

**Artifact.** `results/gate_study_general/{static_gate.csv,static_gate_eval.csv}`

**Caveat.** Rules are interpretable; the comparison bounds the value of learning.

*What this cell does.* Show the best static rule on dev and its eval policy per configuration.

In [ ]:
static = L.load_gate_study_static(); static_eval = L.load_gate_study_static_eval()
display(static.head(5).round(4))
display(static_eval.round(4))

**Reading.** The best dev rule opens feedback when the weakest of the baseline top-5 similarities is
below ~0.95 (dev policy +0.0096, 80% open). On eval it is positive on the calibrated blind
configuration (+0.003) but clearly below the learned gate's recovery on the uncalibrated blind
configurations, where the learned gate closes a sharply harmful subset. A simple rule is a useful
interpretable fallback; the learned gate adds real value only when the harm is concentrated and
multi-feature.

### Does calibration already do the policy's job?

**What we do.** Compare the gate on the uncalibrated (legacy Laplace) prior against the calibrated (EB) prior.

**Artifact.** `results/gate_study_general/{learned_gate.csv,learned_gate_decomposition.csv}`

**Caveat.** Same features; different prior behind the feedback.

*What this cell does.* Show the decomposition for the calibrated blind configuration next to the uncalibrated ones.

In [ ]:
show = decomp[decomp["protocol"] == "blind"].copy()
display(show[["config_key", "dev_threshold", "n_open", "n_closed", "mean_delta_closed", "harm_rate_closed",
              "policy_value", "gain_vs_always_on"]].round(4))

**Reading.** On the uncalibrated prior the gate closes a strongly harmful subset (mean −0.07 to
−0.10) and recovers most of the ceiling. On the calibrated prior (backoff EB) the gate closes tickets
that are 62% no-ops (mean +0.006), so it shaves a small positive tail and loses −0.003. Calibration
and control are therefore **substitutes**: either discipline the prior or gate a raw one. We recommend
calibration as the default (it needs no labels) and keep the policy for settings where the prior
cannot be trusted.

## Conclusion — the control policy

A pre-generation policy is valuable exactly when feedback is unreliable and harm is concentrated.
Under uncalibrated ticket-only feedback the gate closes a sharply harmful subset and recovers 0.54–0.56
of the oracle ceiling; once the prior is calibrated, or under resolution-informed feedback, the closed
set is mostly no-ops and the gate is neutral. The next notebook reports the final results and the claim
ledger; notebook 07 generalises the gate across all signals and replaces sign classification with
expected-value modelling.